# DNAmFitAgeGait

Reproducible conversion of the published DNAmFitAge female and male gait speed regressions into one sex-gated pyaging artifact. The model keeps each coefficient table and reference-median vector independent; the public `female` input selects the corresponding branch.


## Imports and model


In [1]:
import math
import shutil
import subprocess
from pathlib import Path

import pandas as pd
import torch

import pyaging as pya

model = pya.models.DNAmFitAgeGait()


## Curated clock metadata


In [2]:
# ruff: noqa: E501
model.metadata["clock_name"] = "dnamfitagegait"
model.metadata["data_type"] = "DNA methylation"  # Paper: Blood DNA methylation was used to develop the fitness biomarkers.
model.metadata["species"] = "Homo sapiens"  # Paper: The development cohorts were human adult studies (FHS, BLSA, and Budapest).
model.metadata["year"] = 2023
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "McGreevy, K. M., et al. “DNAmFitAge: biological age indicator incorporating physical fitness.” Aging 15(10): 3904–3938 (2023)."
model.metadata["doi"] = "https://doi.org/10.18632/aging.204538"
model.metadata["notes"] = "Sex-gated blood DNAm gait-speed estimator using the published female and male regressions selected by the female input."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["whole blood"]  # Paper: The biomarkers were developed from blood DNAm data.
model.metadata["predicts"] = ["gait speed"]  # Paper: The algorithms generate DNAmGaitspeed, DNAmGripmax, or DNAmVO2max estimates in the corresponding physical-fitness scale.
model.metadata["training_target"] = ["gait speed"]  # Paper: The directly measured fitness parameter was the dependent variable in LASSO regression.
model.metadata["unit"] = ["meters per second"]  # Paper: Gait speed is measured in m/s, grip force in kg, and VO2max in mL/kg/min.
model.metadata["model_type"] = "LASSO regression"  # Paper: Each fitness DNAm biomarker was developed using LASSO penalized regression with ten-fold cross-validation.
model.metadata["platform"] = ["Illumina 450K"]  # Paper: The reported fitness CpG background and fitted loci were on the 450K array.
model.metadata["population"] = "adults"  # Paper: The female and male models were fit separately in adult development cohorts.
model.metadata["journal"] = "Aging"
model.metadata["last_author"] = "Steve Horvath"
model.metadata["n_features"] = 111
model.metadata["citations"] = 99
model.metadata["citations_date"] = "2026-07-05"


## Download and export the published model source

The authors' repository is cloned into this notebook's temporary working directory. R reads the RDS directly and exports only the two coefficient tables and sex-specific median vectors required here.


In [3]:
github_url = "https://github.com/kristenmcgreevy/DNAmFitAge.git"
repository = Path("DNAmFitAge")
if repository.exists():
    shutil.rmtree(repository)
subprocess.run(["git", "clone", github_url, str(repository)], check=True)

r_script = """
DNAmFitnessModels <- readRDS("DNAmFitAge/DNAmFitnessModelsandFitAge_Oct2022.rds")
write.csv(DNAmFitnessModels$Gait_noAge_Females, "Gait_noAge_Females.csv")
write.csv(DNAmFitnessModels$Gait_noAge_Males, "Gait_noAge_Males.csv")
write.csv(DNAmFitnessModels$Female_Medians_All, "FemaleMedians.csv")
write.csv(DNAmFitnessModels$Male_Medians_All, "MaleMedians.csv")
"""
r_script_path = Path("download_dnamfitagegait.r")
r_script_path.write_text(r_script)
subprocess.run(["Rscript", str(r_script_path)], check=True)


Cloning into 'DNAmFitAge'...


CompletedProcess(args=['Rscript', 'download_dnamfitagegait.r'], returncode=0)

## Build the gated model

Feature order is the first-occurrence ordered union of the published female and male feature sequences, followed by `female`. Methylation entries in the public reference vector are NaN sentinels, so each branch applies its own medians during the forward pass.


In [4]:
# ruff: noqa: E501
def ordered_union(*groups):
    return list(dict.fromkeys(feature for group in groups for feature in group))


def load_component(table_name, medians_name):
    table = pd.read_csv(table_name, index_col=0)
    features = table["term"].iloc[1:].tolist()
    coefficients = torch.tensor(table["estimate"].iloc[1:].tolist()).unsqueeze(0)
    intercept = torch.tensor([table["estimate"].iloc[0]])
    component = pya.models.LinearModel(input_dim=len(features))
    component.linear.weight.data = coefficients.float()
    component.linear.bias.data = intercept.float()
    medians = pd.read_csv(medians_name, index_col=0)
    reference_values = medians.loc[1, features].tolist()
    return component, features, reference_values

female_model, female_features, female_references = load_component("Gait_noAge_Females.csv", "FemaleMedians.csv")
male_model, male_features, male_references = load_component("Gait_noAge_Males.csv", "MaleMedians.csv")

model.features = ordered_union(female_features, male_features) + ["female"]
assert len(model.features) == 111
assert "ch.2.105901354F" in male_features
assert "ch.13.39564907R" in male_features
feature_indices = {feature: index for index, feature in enumerate(model.features)}
model.female_model = female_model
model.male_model = male_model
model.female_feature_indices = torch.tensor([feature_indices[feature] for feature in female_features])
model.male_feature_indices = torch.tensor([feature_indices[feature] for feature in male_features])
model.female_reference_values = female_references
model.male_reference_values = male_references
model.female_index = feature_indices["female"]
model.reference_values = [float("nan")] * (len(model.features) - 1) + [1.0]

assert len(model.female_reference_values) == len(female_features)
assert len(model.male_reference_values) == len(male_features)
assert model.reference_values[-1] == 1.0
print({"public_features": len(model.features), "female_features": len(female_features), "male_features": len(male_features)})


{'public_features': 111, 'female_features': 53, 'male_features': 59}


## Verify and save


In [5]:
pya.utils.print_model_details(model)

input_values = torch.full((2, len(model.features)), 0.5, dtype=torch.float64)
input_values[:, model.female_index] = torch.tensor([0.0, 1.0])
model.to(torch.float64).eval()
with torch.no_grad():
    predictions = model(input_values).ravel().tolist()
assert all(math.isfinite(value) for value in predictions)
print({"male_branch": predictions[0], "female_branch": predictions[1]})

torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")



%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'McGreevy, K. M., et al. “DNAmFitAge: biological age indicator '
             'incorporating physical fitness.” Aging 15(10): 3904–3938 (2023).',
 'citations': 99,
 'citations_date': '2026-07-05',
 'clock_name': 'dnamfitagegait',
 'data_type': 'DNA methylation',
 'doi': 'https://doi.org/10.18632/aging.204538',
 'journal': 'Aging',
 'last_author': 'Steve Horvath',
 'model_type': 'LASSO regression',
 'n_features': 111,
 'notes': 'Sex-gated blood DNAm gait-speed estimator using the published '
          'female and male regressions selected by the female input.',
 'platform': ['Illumina 450K'],
 'population': 'adults',
 'predicts': ['gait speed'],
 'research_only': None,
 'species': 'Homo sapiens',
 'tissue': ['whole blood'],
 'training_target': ['gait speed'],
 'unit': ['meters per second'],
 'version': None,
 'yea

## Clear temporary conversion inputs


In [6]:
# ruff: noqa: E501
for path in [repository, r_script_path, Path("FemaleMedians.csv"), Path("MaleMedians.csv"), Path("Gait_noAge_Females.csv"), Path("Gait_noAge_Males.csv")]:
    if path.is_dir():
        shutil.rmtree(path)
    elif path.exists():
        path.unlink()
